# 02 Model Training

This notebook reproduces the main two-step model comparison reported in the repository.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(repo_root)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR

from src.data_pipeline import load_training_set, sanitize_columns, build_feature_matrix
from src.two_step_model import two_step_oof_predictions

try:
    import xgboost as xgb
except Exception:
    xgb = None

try:
    import lightgbm as lgb
except Exception:
    lgb = None

df = sanitize_columns(load_training_set(repo_root / 'data' / 'training_set_257.csv'))
drop_cols = ['Formula', 'E_g_Exp', 'Source', 'Priority', 'pretty_formula', 'Delta_E_g', 'is_metal_exp', 'target_delta']
X, feature_cols = build_feature_matrix(df, drop_cols)
y = df['E_g_Exp'].values
print(X.shape, len(feature_cols))


In [ ]:
models = {
    'Extra Trees': (
        ExtraTreesClassifier(n_estimators=150, max_depth=10, random_state=42),
        ExtraTreesRegressor(n_estimators=200, max_depth=15, random_state=42),
    ),
    'Random Forest': (
        RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42),
        RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42),
    ),
    'Gradient Boosting': (
        GradientBoostingClassifier(n_estimators=150, max_depth=5, random_state=42),
        GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42),
    ),
    'SVM / SVR': (
        make_pipeline(StandardScaler(), SVC(kernel='rbf', probability=True, random_state=42)),
        make_pipeline(StandardScaler(), SVR(kernel='rbf', C=1.0)),
    ),
    'Linear (Ridge/LogReg)': (
        make_pipeline(StandardScaler(), LogisticRegression(random_state=42)),
        make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    ),
}

if xgb is not None:
    models['XGBoost'] = (
        xgb.XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.05, random_state=42, eval_metric='logloss'),
        xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, random_state=42),
    )

if lgb is not None:
    models['LightGBM'] = (
        lgb.LGBMClassifier(n_estimators=150, max_depth=5, learning_rate=0.05, random_state=42, verbose=-1),
        lgb.LGBMRegressor(n_estimators=200, max_depth=6, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1),
    )


In [ ]:
rows = []
for name, (clf, reg) in models.items():
    pred = two_step_oof_predictions(X, y, clf, reg)
    rows.append({'Algorithm': name, 'MAE': mean_absolute_error(y, pred), 'R2': r2_score(y, pred)})

results = pd.DataFrame(rows).sort_values('R2', ascending=False).reset_index(drop=True)
results


In [ ]:
top_row = results.iloc[0].to_dict()
print(top_row)
